In [15]:
import pandas as pd
import numpy as np
import folium
import json

# =========================================================
# 1. 데이터 로드
# =========================================================

file_path = r'C:\py_temp_5\사람인\0511\posting_analysis_table_최종 (7).xlsx'

df = pd.read_excel(file_path)

# =========================================================
# 2. 시각화용 지역 전처리
# =========================================================

df['시각화용_지역'] = (
    df['시각화용_지역']
    .astype(str)
    .str.strip()
)

# ---------------------------------------------------------
# 중복되는 "중구" 예외 처리
# ---------------------------------------------------------

df['시각화용_지역'] = df['시각화용_지역'].replace({

    '서울 중구': '서울중구',
    '부산 중구': '부산중구',
    '대구 중구': '대구중구',
    '인천 중구': '인천중구',
    '대전 중구': '대전중구',
    '울산 중구': '울산중구'
})

# =========================================================
# 3. 지역명 정규화 함수
# =========================================================

def normalize_region(name):

    name = str(name)

    # -----------------------------------------------------
    # 공백 제거
    # -----------------------------------------------------

    name = name.replace(' ', '')

    # -----------------------------------------------------
    # 특별시 / 광역시 / 도 제거
    # -----------------------------------------------------

    remove_words = [

        '서울특별시',
        '부산광역시',
        '대구광역시',
        '인천광역시',
        '광주광역시',
        '대전광역시',
        '울산광역시',

        '서울',
        '부산',
        '대구',
        '인천',
        '광주',
        '대전',
        '울산',

        '경기도',
        '경기'
    ]

    for word in remove_words:
        name = name.replace(word, '')

    # -----------------------------------------------------
    # 경기 주요 도시 구 → 시 통합
    # -----------------------------------------------------

    if '성남시' in name:
        return '성남시'

    if '수원시' in name:
        return '수원시'

    if '고양시' in name:
        return '고양시'

    if '안산시' in name:
        return '안산시'

    if '안양시' in name:
        return '안양시'

    return name.strip()

# =========================================================
# 4. 지역별 통계 집계
# =========================================================

stats = (
    df.groupby('시각화용_지역')
    .agg({
        'posting_quality_score': 'mean',
        'risk_signal_score': 'mean',
        'job': 'count'
    })
    .rename(columns={'job': '공고수'})
    .reset_index()
)

# =========================================================
# 5. region_key 생성
# =========================================================

stats['region_key'] = stats['시각화용_지역'].apply(normalize_region)

# =========================================================
# 6. 중복 region_key 통합
# =========================================================

stats_final = (
    stats.groupby('region_key', as_index=False)
    .agg({
        '공고수': 'sum',
        'posting_quality_score': 'mean',
        'risk_signal_score': 'mean',
        '시각화용_지역': 'first'
    })
)

# =========================================================
# 7. 딕셔너리 변환
# =========================================================

stats_dict = (
    stats_final
    .set_index('region_key')
    .to_dict(orient='index')
)

# =========================================================
# 8. GeoJSON 로드
# =========================================================

geo_path = r'C:\py_temp_5\사람인\skorea-municipalities-2018-geo.json'

with open(geo_path, encoding='utf-8') as f:
    geo_data = json.load(f)

# =========================================================
# 9. GeoJSON 데이터 매칭
# =========================================================

matched_count = 0
unmatched_regions = []

for feature in geo_data['features']:

    geo_name = feature['properties']['name']

    # -----------------------------------------------------
    # 중구 예외 처리
    # -----------------------------------------------------

    geo_code = str(feature.get('id', ''))

    if geo_name == '중구':

        if geo_code.startswith('11'):
            geo_name = '서울중구'

        elif geo_code.startswith('26'):
            geo_name = '부산중구'

        elif geo_code.startswith('27'):
            geo_name = '대구중구'

        elif geo_code.startswith('28'):
            geo_name = '인천중구'

        elif geo_code.startswith('30'):
            geo_name = '대전중구'

        elif geo_code.startswith('31'):
            geo_name = '울산중구'

    # -----------------------------------------------------
    # 정규화
    # -----------------------------------------------------

    geo_key = normalize_region(geo_name)

    match_data = stats_dict.get(geo_key)

    # -----------------------------------------------------
    # 매칭 성공
    # -----------------------------------------------------

    if match_data:

        matched_count += 1

        total_count = int(match_data['공고수'])

        quality = match_data['posting_quality_score']
        risk = match_data['risk_signal_score']

        feature['properties']['match_val'] = np.log1p(total_count)

        feature['properties']['raw_count'] = total_count

        feature['properties']['description'] = f"""
        <div style="font-family:Malgun Gothic; font-size:13px;">
            <b>{geo_name}</b><br><br>

            전체 공고 수:
            <b>{total_count}건</b><br>

            평균 품질 점수:
            <b>{quality:.1f}점</b><br>

            위험 신호 비율:
            <b>{risk:.1%}</b>
        </div>
        """

    # -----------------------------------------------------
    # 매칭 실패
    # -----------------------------------------------------

    else:

        unmatched_regions.append(geo_name)

        feature['properties']['match_val'] = 0

        feature['properties']['raw_count'] = 0

        feature['properties']['description'] = f"""
        <div style="font-family:Malgun Gothic; font-size:13px;">
            <b>{geo_name}</b><br><br>

            데이터 없음
        </div>
        """

# =========================================================
# 10. 매칭 결과 확인
# =========================================================

print("=" * 60)
print(f"매칭 성공 지역 수: {matched_count}")
print("=" * 60)

print("\n===== stats key =====")
print(stats_final['region_key'].unique()[:30])

print("\n===== geo key =====")

for feature in geo_data['features'][:30]:

    geo_name = feature['properties']['name']

    geo_code = str(feature.get('id', ''))

    if geo_name == '중구':

        if geo_code.startswith('11'):
            geo_name = '서울중구'

        elif geo_code.startswith('26'):
            geo_name = '부산중구'

        elif geo_code.startswith('27'):
            geo_name = '대구중구'

        elif geo_code.startswith('28'):
            geo_name = '인천중구'

        elif geo_code.startswith('30'):
            geo_name = '대전중구'

        elif geo_code.startswith('31'):
            geo_name = '울산중구'

    print(normalize_region(geo_name))

print("\n===== 매칭 실패 지역 =====")

for region in unmatched_regions[:30]:
    print(region)

# =========================================================
# 11. 지도 생성
# =========================================================

m = folium.Map(
    location=[36.5, 127.8],
    zoom_start=7,
    tiles='cartodbpositron'
)

# =========================================================
# 12. 컬러 스케일
# =========================================================

max_val = max([
    feature['properties'].get('match_val', 0)
    for feature in geo_data['features']
])

color_scale = folium.branca.colormap.linear.YlGnBu_09.scale(
    0,
    max_val
)

color_scale.caption = '채용공고 분포 (Log Scale)'

# =========================================================
# 13. 스타일 함수
# =========================================================

def style_function(feature):

    val = feature['properties'].get('match_val', 0)

    # 데이터 없음
    if val == 0:

        return {
            'fillColor': '#F2F2F2',
            'color': '#AAAAAA',
            'weight': 0.5,
            'fillOpacity': 0.35
        }

    # 데이터 있음
    return {
        'fillColor': color_scale(val),
        'color': '#666666',
        'weight': 0.7,
        'fillOpacity': 0.75
    }

# =========================================================
# 14. 하이라이트 함수
# =========================================================

def highlight_function(feature):

    return {
        'weight': 2,
        'color': 'black',
        'fillOpacity': 0.9
    }

# =========================================================
# 15. GeoJson 추가
# =========================================================

geojson = folium.GeoJson(

    geo_data,

    style_function=style_function,

    highlight_function=highlight_function,

    tooltip=folium.GeoJsonTooltip(
        fields=['description'],
        aliases=[''],
        labels=False,
        sticky=True
    ),

    name='채용공고 지역 분포'
)

geojson.add_to(m)

# =========================================================
# 16. 컬러 스케일 추가
# =========================================================

color_scale.add_to(m)

# =========================================================
# 17. 제목 추가
# =========================================================

title_html = """
<div style="
    position: fixed;
    top: 15px;
    left: 50%;
    transform: translateX(-50%);
    z-index: 9999;

    background-color: white;
    padding: 12px 22px;

    border-radius: 10px;

    box-shadow: 0 2px 8px rgba(0,0,0,0.2);

    font-family: Malgun Gothic;
">

<b style="font-size:16px;">
채용공고 지역 분포 분석
</b>

<br>

<span style="font-size:12px; color:gray;">
서울 구 단위 + 경기 주요 시 단위 / 로그 스케일
</span>

</div>
"""

m.get_root().html.add_child(folium.Element(title_html))

# =========================================================
# 18. 범례 설명
# =========================================================

legend_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 20px;
    z-index: 9999;

    background-color: white;
    padding: 12px 16px;

    border-radius: 10px;

    box-shadow: 0 2px 8px rgba(0,0,0,0.2);

    font-family: Malgun Gothic;
    font-size: 12px;
">

<b>지도 안내</b><br><br>

색상이 진할수록 공고 수 많음<br>
(Log Scale 적용)<br><br>

회색 지역 = 데이터 없음

</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# =========================================================
# 19. 저장
# =========================================================

save_path = r'C:\py_temp_5\사람인\채용공고_지역분석_최종.html'

m.save(save_path)

print("\nHTML 저장 완료!")
print(save_path)

매칭 성공 지역 수: 94

===== stats key =====
<StringArray>
[     '',   '강남구',   '강동구',   '강서구', '강원강릉시', '강원원주시', '경남김해시', '경남양산시',
 '경남진주시', '경남창원시', '경북구미시', '경북영천시',   '고양시',   '과천시',   '관악구',   '광명시',
   '광산구',   '광진구',   '구로구',   '군위군',   '군포시',   '금천구',   '기장군',   '김포시',
    '남구',   '남동구',  '남양주시',   '달서구',   '달성군',   '대덕구']
Length: 30, dtype: str

===== geo key =====
종로구
중구
용산구
성동구
광진구
동대문구
중랑구
성북구
강북구
도봉구
노원구
은평구
서대문구
마포구
양천구
강서구
구로구
금천구
영등포구
동작구
관악구
서초구
강남구
송파구
강동구
중구
서구
동구
영도구
진구

===== 매칭 실패 지역 =====
강북구
도봉구
노원구
영도구
사하구
금정구
연제구
계양구
강화군
옹진군
세종시
의정부시
동두천시
구리시
오산시
용인시처인구
용인시기흥구
용인시수지구
안성시
포천시
여주시
연천군
가평군
양평군
춘천시
원주시
강릉시
동해시
태백시
속초시

HTML 저장 완료!
C:\py_temp_5\사람인\채용공고_지역분석_최종.html


In [5]:
print(df['시각화용_지역'].dropna().unique()[:30])

<StringArray>
[  '서울 성동구',  '서울 영등포구',   '서울 강서구',   '서울 강남구',   '대전 유성구',   '서울 송파구',
   '제주 제주시',   '서울 서초구', '세종 한누리대로',    '대구 북구',   '경기 김포시',   '경기 부천시',
   '경기 성남시',   '서울 용산구',   '서울 구로구',   '서울 종로구',   '경기 화성시',   '충남 아산시',
   '인천 남동구',   '서울 금천구',   '대구 수성구',   '서울 마포구',   '대구 군위군',    '울산 중구',
   '서울 관악구',       '서울',   '충북 청주시',   '경기 시흥시',    '서울 중구',   '인천 연수구']
Length: 30, dtype: str


In [9]:
for feature in geo_data['features'][:30]:
    print(feature['properties']['name'])

종로구
중구
용산구
성동구
광진구
동대문구
중랑구
성북구
강북구
도봉구
노원구
은평구
서대문구
마포구
양천구
강서구
구로구
금천구
영등포구
동작구
관악구
서초구
강남구
송파구
강동구
중구
서구
동구
영도구
부산진구


In [11]:
print("\n===== stats key =====")
print(stats_final['region_key'].unique()[:30])

print("\n===== geo key =====")

for feature in geo_data['features'][:30]:
    print(normalize_region(feature['properties']['name']))


===== stats key =====
<StringArray>
[ '강원강릉시',  '강원원주시',  '경기고양시',  '경기과천시',  '경기광명시',  '경기광주시',  '경기군포시',
  '경기김포시', '경기남양주시',  '경기부천시',  '경기성남시',  '경기수원시',  '경기시흥시',  '경기안산시',
  '경기안양시',  '경기양주시',  '경기용인시',  '경기의왕시',  '경기이천시',  '경기파주시',  '경기평택시',
  '경기하남시',  '경기화성시',  '경남김해시',  '경남양산시',  '경남진주시',  '경남창원시',  '경북구미시',
  '경북영천시',  '광주광산구']
Length: 30, dtype: str

===== geo key =====
종로구
중구
용산구
성동구
광진구
동대문구
중랑구
성북구
강북구
도봉구
노원구
은평구
서대문구
마포구
양천구
강서구
구로구
금천구
영등포구
동작구
관악구
서초구
강남구
송파구
강동구
중구
서구
동구
영도구
부산진구


In [12]:
print(geo_data['features'][0]['properties'])

{'name': '종로구', 'base_year': '2018', 'name_eng': 'Jongno-gu', 'code': '11010', 'match_val': 0, 'raw_count': 0, 'description': '\n        <div style="font-family:Malgun Gothic; font-size:13px;">\n            <b>종로구</b><br><br>\n\n            데이터 없음\n        </div>\n        '}
